In [1]:
import requests
from collections import defaultdict

RPC_CHAIN_URL = "http://192.168.4.80:26657"
REST_CHAIN_URL = "http://192.168.4.80:1317"


def get_current_epoch_data(height=None):
    headers = {}
    if height is not None:
        headers["X-Cosmos-Block-Height"] = str(height)

    res = requests.get(f"{REST_CHAIN_URL}/productscience/inference/inference/current_epoch_group_data", headers=headers)
    return res.json()


def get_group(group_id, height=None):
    headers = {}
    if height is not None:
        headers["X-Cosmos-Block-Height"] = str(height)

    res = requests.get(f"{REST_CHAIN_URL}/cosmos/group/v1/group_members/{group_id}?pagination.limit=10000", headers=headers)
    return res.json()

# PoC 134


## Getting generated batches

In [2]:
res = requests.get(f"{REST_CHAIN_URL}/productscience/inference/inference/poc_batches_for_stage/2073749").json()

In [3]:
participant_to_batches = defaultdict(list)


for participant in res["poc_batch"]:
    batch = participant["poc_batch"]
    participant_to_batches[participant["participant"]].append(batch)

def get_stats(participant):
    batches = participant_to_batches[participant]
    all_nonces = [nonce for batch_list in batches for batch in batch_list for nonce in batch["nonces"]]
    all_distances = [dist for batch_list in batches for batch in batch_list for dist in batch["dist"]]
    mlnode_to_batches = defaultdict(list)
    for batch_list in batches:
        for batch in batch_list:
            mlnode_to_batches[batch["node_id"]].extend(batch["nonces"])

    return all_nonces, all_distances, mlnode_to_batches


participant_to_stats = defaultdict(dict)
for participant in res["poc_batch"]:
    all_nonces, all_distances, mlnode_to_batches = get_stats(participant["participant"])
    participant_to_stats[participant["participant"]] = {
        "all_nonces": all_nonces,
        "all_distances": all_distances,
        "mlnode_to_batches": mlnode_to_batches
    }

## Getting validations

In [4]:
val_res = requests.get(f"{REST_CHAIN_URL}/productscience/inference/inference/poc_validations_for_stage/2073749").json()

participant_to_validation = defaultdict(lambda: defaultdict(list))
for validation in val_res["poc_validation"]:
    participant = validation["participant"]
    for validation in validation["poc_validation"]:
        to_validate = validation["participant_address"]
        assert to_validate == participant
        validator = validation["validator_participant_address"]
        fraud_detected = bool(validation["fraud_detected"])

        if fraud_detected:
            participant_to_validation[participant]["invalid"].append(validator)
        else:
            participant_to_validation[participant]["valid"].append(validator)    

In [5]:
current_epoch_data = get_current_epoch_data(2073934)
participant_to_weight = {x["member_address"]: x["weight"] for x in current_epoch_data["epoch_group_data"]["validation_weights"]}


group = get_group(680, 2073934)
participant_to_weight = {x["member"]["address"]: x["member"]["weight"] for x in group["members"]}

In [6]:
total_weight = sum(int(x) for x in participant_to_weight.values())
total_weight

8690384

## Total weight of validators per each participant 

In [7]:
counter = 0
for participant, validators in sorted(participant_to_validation.items(), key=lambda x: len(participant_to_stats[x[0]]["all_nonces"]), reverse=True):
    total_valid = sum(int(participant_to_weight.get(validator, 0)) for validator in validators["valid"])
    total_invalid = sum(int(participant_to_weight.get(validator, 0)) for validator in validators["invalid"])
    print(counter, participant, total_valid, total_invalid)
    counter += 1


0 gonka1lrls7q8nu8rhjgchctswfsjlnjh84vwycw3jgt 4451354 0
1 gonka1w6wwv3wq25p8qge4lqsnfzs8lsd3s8ty6au65p 4335742 0
2 gonka145qtr0h90fvz88klshvl4r4htw4thdgcvpey3g 4467160 0
3 gonka1aun6f73uq2r5fujk38xe0tww980r6a0lz6a45g 4457592 0
4 gonka16tru27h865g77cfhsyfpepcjlqlgavl4dc0a0t 4468910 0
5 gonka155cnj622zfdl64f23ljmk2tzv7tewl6fp4m2hl 4161688 0
6 gonka135yst4re34z3rlfquhknqxvnaj35pjwu0mw622 4300583 0
7 gonka1p2959dx973hd57qsalxvesrcv649296x90ry76 4429225 0
8 gonka14c0vannxsak0qq4lpmr5qkn40gcfmcvnyu4z2r 4409173 0
9 gonka172esz53695ykvv038y8mvdfc6mkk2q0dgfmjks 4357177 0
10 gonka1mwwzgd5qztcjddg2pwej0xw8xywqr5l4san6ee 4420658 0
11 gonka14fcf7yqtu6543y7qmhqf62c6xks9nuyprws47l 0 4466064
12 gonka1r9zaxcvn5rysp09hrlla6k8ltssuhtpl07wtqu 4134087 0
13 gonka1y95qr73kms3zv5ju0kxtu58ksxdyrkyz8m0430 4360537 0
14 gonka166wgzjsx5hm4cxc58uc42watkgcppgv80yts48 4418730 0
15 gonka15y9q9je0hs0uhq39v37g9yprhxhj8rqu5cqc58 4386047 0
16 gonka1e2wq9deve5zk6znff34c0x9xdklt0qs6u43x4x 4420121 0
17 gonka1f7d43z2qpcv07hu

In [8]:
all_participant_with_weight = [participant for participant in participant_to_stats if int(participant_to_weight.get(participant, 0)) > 0]

print(
    f"Participant who had weight in epoch 133 and participated in PoC for epoch 134:\n"
    f" length: {len(all_participant_with_weight)}\n"
    f" weight: {sum(int(participant_to_weight.get(validator, 0)) for validator in all_participant_with_weight)}\n"
)

Participant who had weight in epoch 133 and participated in PoC for epoch 134:
 length: 338
 weight: 7284757



In [9]:
all_ever_voted = set()

for validators in participant_to_validation.values():
    all_ever_voted.update(validators["valid"])
    all_ever_voted.update(validators["invalid"])


all_ever_voted_with_weight = [participant for participant in all_ever_voted if int(participant_to_weight.get(participant, 0)) > 0]

print(
    f"Participant who had weight in epoch 133 and validated PoC for epoch 134:\n"
    f" length: {len(all_ever_voted_with_weight)}\n"
    f" weight: {sum(int(participant_to_weight.get(validator, 0)) for validator in all_ever_voted_with_weight)}\n"
)

Participant who had weight in epoch 133 and validated PoC for epoch 134:
 length: 301
 weight: 4643277



In [10]:
not_voted = []
for previous_participant in all_participant_with_weight:
    if previous_participant not in participant_to_weight:
        continue

    if previous_participant not in all_ever_voted_with_weight:
        not_voted.append(previous_participant)
        continue

print(
    len(not_voted),
    sum(int(participant_to_weight.get(validator, 0)) for validator in not_voted)
)

print(
    f"Participant who had weight in epoch 133 and NOT validated PoC for epoch 134:\n"
    f" length: {len(not_voted)}\n"
    f" weight: {sum(int(participant_to_weight.get(validator, 0)) for validator in not_voted)}\n"
)

for idx, participant in enumerate(not_voted):
    print(f"{idx:02d}: {participant}")


37 2641480
Participant who had weight in epoch 133 and NOT validated PoC for epoch 134:
 length: 37
 weight: 2641480

00: gonka1z4ldfav9tl7x3w9aqfry89zd0kt7sa2lhff6te
01: gonka1yquunstmnn35tuc6wsrlv020ul2rnuyldqkt37
02: gonka1tdgnyws5p4ddg2wsg08tmdwggfxpp6e4ev29yc
03: gonka1e2wq9deve5zk6znff34c0x9xdklt0qs6u43x4x
04: gonka1z9wz95l4eu2ltwvavy4m7w5qjvyac3ktt5zhs2
05: gonka1sg2uj8u59hjkzh6qwfsgg0n0w63ux35my80h4a
06: gonka1qfq964gewr83f4vh26qz25hl8na7kmvy93j3el
07: gonka138qmc42pq75mxf3r3fh9k35ukvj9frmvuc7f0p
08: gonka18n9rq8q9gq0guvrm2mzk7hd2hvtf6cf6k6ul44
09: gonka15ye48c3rp57hwt9t6tzc6h0t5sh88zvj0yznwn
10: gonka166wgzjsx5hm4cxc58uc42watkgcppgv80yts48
11: gonka14vp3katk6685avqjxhwrn6cckrpgjrf5vjz8jm
12: gonka1ev28hzcw4tduvnuk0avlckq33cusgf3n3gr76k
13: gonka17kua4qvnfmedl4atk2e03gjlgm42qxckef6uhw
14: gonka1hk6css6jcxwcjnft04fwmnww67u8kuw0865gv3
15: gonka15y9q9je0hs0uhq39v37g9yprhxhj8rqu5cqc58
16: gonka172eahn496yyps4fap74fv4y843n4ghl43atmkk
17: gonka135yst4re34z3rlfquhknqxvnaj35pjwu0mw622
